# Given a sim, investigate flux through the metabolic network by plotting a sankey diagram of fluxes through reactions. This is a more detailed version of the reaction usage plot, which only plots the number of reactions used and kinetic target scatter.

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast, Any
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from yaml import emit

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def sankey_metabolite_flux(
    met_of_interest,
    fba: dict,
    metabolism: object,
    *,
    timestep=None,
    agg: str = "mean",
    top_n: int = 12,
    min_abs_contrib: float | None = None,
    title: str | None = None,
):
    """
    Sankey for ONE metabolite using user's get_subset_S().

    Returns
    -------
    fig : plotly.graph_objects.Figure
    contrib_table : pd.DataFrame
    """
    S = metabolism.stoichiometry.copy()
    reaction_names = metabolism.reaction_names
    metabolites = metabolism.metabolite_names
    S = pd.DataFrame(S, index=metabolites, columns=reaction_names)
    flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
    # --- user's helper ---
    def get_subset_S(S, met_of_interest):
        S_met = S.loc[met_of_interest, :]
        S_met = S_met.loc[:, ~np.all(S_met == 0, axis=0)]
        return S_met, S_met.columns

    # Normalize met input
    if isinstance(met_of_interest, str):
        met_list = [met_of_interest]
    else:
        met_list = list(met_of_interest)

    if len(met_list) != 1:
        raise ValueError("Please pass exactly one metabolite name (string or single-item list).")
    met = met_list[0]

    # Get stoich row subset (1 x rxns) and participating rxns
    S_met, rxns = get_subset_S(S, [met])  # returns DataFrame with 1 row
    s = S_met.loc[met]                    # Series indexed by rxns

    # --- pick a single flux vector v (Series indexed by rxns) ---
    if isinstance(flux, pd.DataFrame):
        # restrict to participating rxns first to avoid reindex noise
        flux_sub = flux.loc[:, rxns]

        if timestep is not None:
            v = flux_sub.loc[timestep]
        else:
            if agg == "mean":
                v = flux_sub.mean(axis=0)
            elif agg == "sum":
                v = flux_sub.sum(axis=0)
            elif agg == "median":
                v = flux_sub.median(axis=0)
            elif agg == "max":
                v = flux_sub.max(axis=0)
            elif agg == "min":
                v = flux_sub.min(axis=0)
            else:
                raise ValueError(f"Unknown agg='{agg}'")
    elif isinstance(flux, pd.Series):
        v = flux.reindex(rxns).fillna(0.0)
    else:
        raise TypeError("flux must be a pandas Series or DataFrame")

    # Signed contribution to d(met)/dt
    contrib = s * v
    abs_contrib = contrib.abs()

    contrib_table = pd.DataFrame(
        {
            "stoich": s,
            "flux": v,
            "contrib": contrib,
            "abs_contrib": abs_contrib,
        }
    ).sort_values("abs_contrib", ascending=False)

    if min_abs_contrib is not None:
        contrib_table = contrib_table[contrib_table["abs_contrib"] >= float(min_abs_contrib)]

    # Producers/consumers by SIGN of contribution
    producers = contrib_table[contrib_table["contrib"] > 0].head(top_n)
    consumers = contrib_table[contrib_table["contrib"] < 0].head(top_n)

    if producers.empty and consumers.empty:
        raise ValueError(
            f"No nonzero contributions found for '{met}' "
            f"(after filtering). Try a different timestep/agg or lower min_abs_contrib."
        )

    # --- Sankey nodes ---
    prod_names = producers.index.tolist()
    cons_names = consumers.index.tolist()
    nodes = prod_names + [met] + cons_names
    idx = {name: i for i, name in enumerate(nodes)}
    met_idx = idx[met]

    # --- Sankey links ---
    sources, targets, values, link_labels = [], [], [], []

    for r, row in producers.iterrows():
        sources.append(idx[r])
        targets.append(met_idx)
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} produces {met}: |S*v|={row['abs_contrib']:.3g}")

    for r, row in consumers.iterrows():
        sources.append(met_idx)
        targets.append(idx[r])
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} consumes {met}: |S*v|={row['abs_contrib']:.3g}")

    # --- Plot ---
    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=12,
                    thickness=16,
                    line=dict(width=0.5),
                    label=nodes,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    label=link_labels,
                ),
            )
        ]
    )

    if title is None:
        if isinstance(flux, pd.DataFrame) and timestep is not None:
            title = f"{met}: production/consumption contributions (t={timestep})"
        elif isinstance(flux, pd.DataFrame):
            title = f"{met}: production/consumption contributions ({agg} over time)"
        else:
            title = f"{met}: production/consumption contributions"

    fig.update_layout(title=title, template="plotly_white", height=650)
    return fig, contrib_table


In [13]:
# import sim
time_num = 600
date = '2026-01-22'
experiment_name = 'homeostatic_only'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [18]:
met_of_interest = 'CMP[c]'
fig = sankey_metabolite_flux(
    met_of_interest=met_of_interest,
    fba=fba,
    metabolism=metabolism,
    agg="mean",
    top_n=10,
    min_abs_contrib=1e-6,
    title=f"{met_of_interest} production/consumption contributions (mean over time)",
)[0]
fig.show()

In [22]:
# import sim
time_num = 600
date = '2026-02-24'
experiment_name = 'homeostatic_only_after_PR'
condition = 'basal'
experiment_type = 'objective_weight'

fba_pr, bulk_pr, metabolism_pr, output_pr = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [32]:
met_of_interest = 'CMP[c]'
fig = sankey_metabolite_flux(
    met_of_interest=met_of_interest,
    fba=fba_pr,
    metabolism=metabolism_pr,
    agg="mean",
    top_n=10,
    min_abs_contrib=1e-6,
    title=f"{met_of_interest} production/consumption contributions (mean over time)",
)[0]
fig.show()